In [ ]:
import tkinter as tk
from PIL import Image, ImageTk
import random
import heapq
#import time

thief_move_delay = 300   # same as guard (300ms)
last_move_time = 0  #to rmv same speed rmv this 2 lines

# Create root
root = tk.Tk()
root.title("AI Guard vs Thief")
root.geometry("1000x600")

canvas = tk.Canvas(root, width=1000, height=600, bg="black")
canvas.pack()

# Game title
canvas.create_text(
    500, 150,
    text="AI GUARD VS THIEF",
    fill="white",
    font=("Arial", 40, "bold")
)

# Start button
start_button = tk.Button(
    root,
    text="START GAME",
    font=("Arial", 16),
    bg="green",
    fg="white"
)

start_button.place(x=440, y=250)

# Instructions system
instructions_visible = False
instruction_text = None

def toggle_instructions():
    global instructions_visible, instruction_text
    

    if not instructions_visible:
        instruction_text = canvas.create_text(
            650, 470,
            text="1. Move thief by Arrow Keys \n         (Right, Left, Up & Down) \n2. Collect all the coins \n3. Avoid the guard \n4. Reach the door to win \n\nRead all the instructions? \n       Best of Luck.",
            fill="white",
            font=("Arial",18),
            justify="left"
        )
        instructions_visible = True

    else:
        canvas.delete(instruction_text)
        instructions_visible = False


# Instructions button
instructions_button = tk.Button(
    root,
    text="INSTRUCTIONS",
    font=("Arial",16),
    bg="green",
    fg="white",
    command=toggle_instructions
)

instructions_button.place(x=440, y=300)

In [4]:
# Tile setup
TILE_SIZE = 50
GRID_WIDTH = 20
GRID_HEIGHT = 12

# Map grid: 0=empty, 1=obstacle
grid = [[0]*GRID_WIDTH for _ in range(GRID_HEIGHT)]

# Positions
thief_pos = [0,0]
guard_pos = [14,10]
door_pos =  (18,10)



def generate_coins(num_coins):
    new_coins = []

    while len(new_coins) < num_coins:
        x = random.randint(0, GRID_WIDTH - 1)
        y = random.randint(0, GRID_HEIGHT - 1)

        # Avoid obstacles, thief start, guard start, and door
        if (
            grid[y][x] == 0 and
            [x, y] != thief_pos and
            [x, y] != guard_pos and
            (x, y) != door_pos and
            all(c["pos"] != [x, y] for c in new_coins)
        ):
            new_coins.append({"id": None, "pos": [x, y]})

    return new_coins

coins = generate_coins(10)

# Load images
thief_img = ImageTk.PhotoImage(Image.open("images/thief.png").resize((40,60)), master=root)
guard_img = ImageTk.PhotoImage(Image.open("images/guard.png").resize((40,60)), master=root)
guardf_img = ImageTk.PhotoImage(Image.open("images/guardf.png").resize((40,60)), master=root)
coin_img = ImageTk.PhotoImage(Image.open("images/coin.png").resize((25,25)), master=root)
tree_img = ImageTk.PhotoImage(Image.open("images/tree.png").resize((50,50)), master=root)
wall_img = ImageTk.PhotoImage(Image.open("images/wall.png").resize((50,50)), master=root)
door_img = ImageTk.PhotoImage(Image.open("images/door.png").resize((100,80)), master=root)
win_img = ImageTk.PhotoImage(Image.open("images/win.png").resize((300,250)), master=root)
caught_img = ImageTk.PhotoImage(Image.open("images/caught.png").resize((300,300)), master=root)

images = [thief_img, guard_img, coin_img, tree_img, wall_img, door_img]

def generate_obstacles(num_trees, num_walls):
    new_trees = []
    new_walls = []

    occupied = set()

    # Reserve positions (cannot place obstacles here)
    occupied.add(tuple(thief_pos))
    occupied.add(tuple(guard_pos))
    occupied.add(door_pos)

    # Generate trees
    while len(new_trees) < num_trees:
        x = random.randint(0, GRID_WIDTH - 1)
        y = random.randint(0, GRID_HEIGHT - 1)

        if (x, y) not in occupied:
            new_trees.append((x, y))
            occupied.add((x, y))

    # Generate walls
    while len(new_walls) < num_walls:
        x = random.randint(0, GRID_WIDTH - 1)
        y = random.randint(0, GRID_HEIGHT - 1)

        if (x, y) not in occupied:
            new_walls.append((x, y))
            occupied.add((x, y))

    return new_trees, new_walls

#Start game function
def start_game():
    
    global trees, walls, grid
    global coins
    
    trees, walls = generate_obstacles(5, 5)

    # Reset grid
    grid = [[0]*GRID_WIDTH for _ in range(GRID_HEIGHT)]

    # Mark obstacles
    for x, y in trees + walls:
        grid[y][x] = 1
    
    coins = generate_coins(total_coins)
    global thief, guard
    global score_text

    start_button.destroy()
    instructions_button.destroy()
    canvas.delete("all")
    for widget in root.winfo_children():
        if isinstance(widget, tk.Button):
            widget.place_forget()
    score_text = canvas.create_text(
        500,
        20,
        text=f"Coins: {score}/{10}",
        fill="yellow",
        font=("Arial",18,"bold")
    )
    # Create thief
    thief = canvas.create_image(
        thief_pos[0]*TILE_SIZE,
        thief_pos[1]*TILE_SIZE,
        image=thief_img,
        anchor="nw"
    )
    # Create guard
    guard = canvas.create_image(
        guard_pos[0]*TILE_SIZE,
        guard_pos[1]*TILE_SIZE,
        image=guard_img,
        anchor="nw"
    )
    # Door
    canvas.create_image(
        door_pos[0]*TILE_SIZE,
        door_pos[1]*TILE_SIZE,
        image=door_img,
        anchor="nw"
    )
    # Tree obs
    for x,y in trees:
        canvas.create_image(x*TILE_SIZE, y*TILE_SIZE, image=tree_img, anchor="nw")
    # Wall obs
    for x,y in walls:
        canvas.create_image(x*TILE_SIZE, y*TILE_SIZE, image=wall_img, anchor="nw")
    # Coins
    for coin in coins:
        x,y = coin["pos"]
        coin["id"] = canvas.create_image(x*TILE_SIZE, y*TILE_SIZE, image=coin_img, anchor="nw")
    move_guard()

# Connect button
start_button.config(command=start_game)
game_over = False
score = 0
total_coins = len(coins)

# 7️⃣ A* Pathfinding functions
def astar(start, goal):
    open_set = []
    heapq.heappush(open_set, (0,start))
    came_from = {}
    g_score = {tuple(start):0}
    f_score = {tuple(start):heuristic(start, goal)}

    while open_set:
        current = heapq.heappop(open_set)[1]
        if current == goal:
            return reconstruct_path(came_from, current)
        for dx,dy in [(0,1),(1,0),(0,-1),(-1,0)]:
            neighbor = [current[0]+dx, current[1]+dy]
            if 0<=neighbor[0]<GRID_WIDTH and 0<=neighbor[1]<GRID_HEIGHT:
                if grid[neighbor[1]][neighbor[0]] == 1:
                    continue
                tentative_g = g_score[tuple(current)] + 1
                if tuple(neighbor) not in g_score or tentative_g < g_score[tuple(neighbor)]:
                    came_from[tuple(neighbor)] = current
                    g_score[tuple(neighbor)] = tentative_g
                    f_score[tuple(neighbor)] = tentative_g + heuristic(neighbor, goal)
                    heapq.heappush(open_set, (f_score[tuple(neighbor)], neighbor))
    return []

def heuristic(a,b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])
def reconstruct_path(came_from, current):
    path = []
    while tuple(current) in came_from:
        path.append(current)
        current = came_from[tuple(current)]
    path.reverse()
    return path

# ----------------------------
# 8️⃣ Guard movement
def move_guard():
    global guard_pos, game_over

    if game_over:
        return

    if thief_pos == guard_pos:
        game_over = True
        game_over_screen("GUARD WINS!", caught_img)
        return
    
    path = astar(guard_pos, thief_pos)

    if path:
        next_step = path[0]
        guard_pos = next_step
        canvas.coords(guard,guard_pos[0]*TILE_SIZE,guard_pos[1]*TILE_SIZE)
        # Change guard direction depending on thief position
        if thief_pos[0] > guard_pos[0]:
            canvas.itemconfig(guard, image=guardf_img)
        else:
            canvas.itemconfig(guard, image=guard_img)

    root.after(300, move_guard)
    
def game_over_screen(message, img=None):
    canvas.delete("all")
    canvas.create_rectangle(0,0,1000,800,fill="black")

    canvas.create_text(
        500,
        150,  # higher up to make space for the image
        text=message,
        fill="white",
        font=("Arial",40,"bold")
    )

    # If image is provided, show it below the message
    if img:
        canvas.image= img
        canvas.create_image(
            500,  # center x
            300,  # y position for image
            image=img,
        
        )

    # Restart button
    restart_btn = tk.Button(
        root,
        text="RESTART",
        font=("Arial",18),
        bg="green",
        fg="white",
        command=restart_game
    )
    restart_btn.place(x=400, y=500)

    # Exit button
    exit_btn = tk.Button(
        root,
        text="EXIT",
        font=("Arial",18),
        bg="red",
        fg="white",
        command=root.destroy
    )
    exit_btn.place(x=550, y=500)
    
# 9️⃣ Thief movement & coin collection

    
def restart_game():
        
    global thief_pos, guard_pos, coins, score, game_over

    thief_pos = [0,0]
    guard_pos = [14,10]

    score = 0
    game_over = False

    trees, walls = generate_obstacles(5,5)
    coins = generate_coins(total_coins)
    
    start_game()
    
    
def move_thief(event):
    global thief_pos, game_over, score  #, last_move_time #from lst-mv-tm to

    # ⛔ SPEED CONTROL (same speed as guard)
    #current_time = int(time.time() * 1000)
    #if current_time - last_move_time < thief_move_delay:
        #return
    #last_move_time = current_time #if u dont wanna same speed cut to this line
    
    if game_over:
        return

    dx = dy = 0

    if event.keysym == "Up":
        dy = -1
    elif event.keysym == "Down":
        dy = 1
    elif event.keysym == "Left":
        dx = -1
    elif event.keysym == "Right":
        dx = 1

    new_x = thief_pos[0] + dx
    new_y = thief_pos[1] + dy
    
    
    # Check boundary and obstacles
    if 0 <= new_x < GRID_WIDTH and 0 <= new_y < GRID_HEIGHT and grid[new_y][new_x] == 0:
        thief_pos = [new_x, new_y]
        canvas.coords(thief, new_x*TILE_SIZE, new_y*TILE_SIZE)

    # Coin collection
    collected = []
    for coin in coins:
        if coin["pos"] == thief_pos:
            canvas.delete(coin["id"])
            collected.append(coin)
    

    for coin in collected:
        coins.remove(coin)
        score += 1
    canvas.itemconfig(score_text, text=f"Coins: {score}/{total_coins}")

    # ✅ Thief win condition
    # if  thief_pos == list(door_pos) and len(coins) == 0:
    if  len(coins) == 0 and thief_pos == list(door_pos):
        game_over = True
        game_over_screen("THIEF WINS!", win_img)

    

# ----------------------------
# 10️⃣ Bind keys & start guard
root.bind("<KeyPress>", move_thief)
root.mainloop()